In [11]:
# 1. Setup & Imports
!pip install -q yfinance prophet transformers fugashi unidic_lite ipadic statsmodels scikit-learn beautifulsoup4

import os
import sqlite3
import warnings
import requests
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import torch
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from prophet import Prophet
from transformers import pipeline
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, MultiHeadAttention, LayerNormalization, Dropout, GlobalAveragePooling1D, Concatenate
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
%matplotlib inline

tf_gpus = tf.config.list_physical_devices('GPU')
if tf_gpus:
    try:
        for gpu in tf_gpus: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e: print(f"GPU config error: {e}")

In [12]:
# 2. Constants & Configuration
NIKKEI_TICKER = "1571.T"
GOLD_TICKER = "GC=F"
DB_NAME = "market_data.db"
LOOKBACK_WINDOW = 60
PREDICTION_HORIZONS = [2, 3, 5, 7, 11, 13, 17]
CATEGORIES = ["金融政策", "経済指標", "企業業績", "市場動向", "地政学"]
TOPIC_COLS = ["monetary_policy", "economic_indicators", "corporate_earnings", "market_trend", "geopolitics"]
RSS_FEEDS = [
    "https://news.yahoo.co.jp/rss/categories/business.xml",
    "https://finance.yahoo.co.jp/rss/news/market.xml"
]

In [13]:
# 3. Database Utility
def setup_database():
    conn = sqlite3.connect(DB_NAME)
    cols = ", ".join([f"{c} REAL DEFAULT 0" for c in TOPIC_COLS])
    conn.execute(f'CREATE TABLE IF NOT EXISTS daily_events (date TEXT PRIMARY KEY, avg_sentiment REAL, {cols})')
    conn.execute('''CREATE TABLE IF NOT EXISTS news_articles (id INTEGER PRIMARY KEY AUTOINCREMENT, date TEXT, title TEXT, sentiment_score REAL, topic TEXT, url TEXT)''')
    conn.commit(); conn.close()

setup_database()

In [14]:
# 4. News Scraping & LLM Analysis Pipeline
def fetch_and_analyze_news_v3():
    print("Starting News Scraping & LLM Analysis...")
    # Initialize NLP Pipelines
    # Using mDeBERTa-v3 for Zero-Shot Classification (Topic categorizing)
    # Using BERT-base-multilingual-uncased-sentiment for Sentiment analysis
    device = 0 if torch.cuda.is_available() else -1
    sentiment_analyzer = pipeline("sentiment-analysis", model="bert-base-multilingual-uncased-sentiment", device=device)
    classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=device)
    
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    today_str = datetime.now().strftime('%Y-%m-%d')
    
    news_results = []
    for url in RSS_FEEDS:
        try:
            resp = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(resp.content, "xml")
            items = soup.find_all("item")
            for item in items:
                title = item.title.text
                link = item.link.text
                
                # LLM Analysis
                # 1. Sentiment Score (Map 1-5 stars to -1 to 1)
                sent_out = sentiment_analyzer(title)[0]
                star_rating = int(sent_out['label'][0])
                sentiment_score = (star_rating - 3) / 2 # Normalize to range [-1, 1]
                
                # 2. Topic Classification
                class_out = classifier(title, candidate_labels=CATEGORIES)
                top_topic = class_out['labels'][0]
                topic_scores = {CATEGORIES[i]: class_out['scores'][i] for i in range(len(CATEGORIES))}
                
                news_results.append({
                    'date': today_str,
                    'title': title,
                    'sentiment': sentiment_score,
                    'topic': top_topic,
                    'scores': topic_scores,
                    'url': link
                })
        except Exception as e: print(f"Error fetching {url}: {e}")
    
    # Save to Database
    if news_results:
        conn = sqlite3.connect(DB_NAME)
        for res in news_results:
            conn.execute("INSERT INTO news_articles (date, title, sentiment_score, topic, url) VALUES (?,?,?,?,?)", 
                         (res['date'], res['title'], res['sentiment'], res['topic'], res['url']))
        
        # Update daily_events table
        avg_sent = np.mean([r['sentiment'] for r in news_results])
        avg_topics = {col: np.mean([r['scores'][CATEGORIES[i]] for r in news_results]) for i, col in enumerate(TOPIC_COLS)}
        
        topic_vals = [avg_topics[c] for c in TOPIC_COLS]
        query = f"INSERT OR REPLACE INTO daily_events (date, avg_sentiment, {', '.join(TOPIC_COLS)}) VALUES (?,?,{','.join(['?']*len(TOPIC_COLS))})"
        conn.execute(query, (today_str, avg_sent, *topic_vals))
        
        conn.commit(); conn.close()
        print(f"Processed and stored {len(news_results)} news articles.")
    else: print("No news found.")

fetch_and_analyze_news_v3()

Starting News Scraping & LLM Analysis...


OSError: bert-base-multilingual-uncased-sentiment is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
# 5. Data Loading & Preparation
def load_combined_data(ticker):
    df_stock = yf.download(ticker, start="2000-01-01")
    if isinstance(df_stock.columns, pd.MultiIndex): df_stock.columns = df_stock.columns.get_level_values(0)
    df_stock = df_stock.reset_index()
    df_stock['Date_str'] = df_stock['Date'].dt.strftime('%Y-%m-%d')
    
    conn = sqlite3.connect(DB_NAME)
    df_events = pd.read_sql_query("SELECT * FROM daily_events", conn)
    conn.close()
    
    df = pd.merge(df_stock, df_events, left_on='Date_str', right_on='date', how='left').fillna(0.0)
    # Add a fallback for empty event data (fill with zeros for older dates)
    return df

In [ ]:
# 6. Cross-Attention Model Definition
def build_cross_attention_model(lookback, event_dim):
    price_input = Input(shape=(lookback, 1), name='price_input')
    event_input = Input(shape=(lookback, event_dim), name='event_input')
    
    p_encoding = LSTM(128, return_sequences=True)(price_input)
    p_encoding = LayerNormalization()(p_encoding)
    
    e_encoding = LSTM(128, return_sequences=True)(event_input)
    e_encoding = LayerNormalization()(e_encoding)
    
    # Price attends to Event Context
    attended = MultiHeadAttention(num_heads=4, key_dim=64)(query=p_encoding, value=e_encoding)
    combined = Concatenate()([p_encoding, attended])
    x = Dropout(0.2)(combined)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    output = Dense(1, name='output')(x)
    
    model = Model(inputs=[price_input, event_input], outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
# 7. Multi-Asset Multi-Horizon Pipeline
def run_multi_horizon_forecasting(ticker, horizons):
    print(f"\n--- Forecasting Cycle for {ticker} ---")
    df = load_combined_data(ticker)
    scaler_p = MinMaxScaler(); scaler_e = MinMaxScaler()
    p_data = scaler_p.fit_transform(df[['Close']].values)
    e_data = scaler_e.fit_transform(df[TOPIC_COLS].values)
    
    predictions = {}
    for h in horizons:
        X_p, X_e, Y = [], [], []
        # Build sequences
        for i in range(len(p_data) - LOOKBACK_WINDOW - h):
            X_p.append(p_data[i:i+LOOKBACK_WINDOW])
            X_e.append(e_data[i:i+LOOKBACK_WINDOW])
            Y.append(p_data[i+LOOKBACK_WINDOW+h-1])
        
        if not X_p: continue
        X_p, X_e, Y = np.array(X_p), np.array(X_e), np.array(Y)
        model = build_cross_attention_model(LOOKBACK_WINDOW, len(TOPIC_COLS))
        model.fit([X_p, X_e], Y, epochs=10, batch_size=32, verbose=0, callbacks=[EarlyStopping(patience=3)])
        
        # Predict latest
        last_xp = p_data[-LOOKBACK_WINDOW:].reshape(1, LOOKBACK_WINDOW, 1)
        last_xe = e_data[-LOOKBACK_WINDOW:].reshape(1, LOOKBACK_WINDOW, len(TOPIC_COLS))
        pred_scaled = model.predict([last_xp, last_xe], verbose=0)
        predictions[h] = scaler_p.inverse_transform(pred_scaled)[0,0]
    
    return df, predictions

gold_df, gold_preds = run_multi_horizon_forecasting(GOLD_TICKER, PREDICTION_HORIZONS)
nikkei_df, nikkei_preds = run_multi_horizon_forecasting(NIKKEI_TICKER, PREDICTION_HORIZONS)

In [ ]:
# 8. Forecast Summary & Visualization
def plot_forecasts(ticker, df, preds):
    plt.figure(figsize=(14, 7))
    plt.plot(df['Date'].tail(300), df['Close'].tail(300), label='Historical Price', color='blue', alpha=0.6)
    for h, val in preds.items():
        future_date = df['Date'].iloc[-1] + timedelta(days=h)
        plt.scatter(future_date, val, label=f'T+{h} Forecast: {val:,.2f}', marker='*', s=150)
    plt.title(f"{ticker} Price Forecast - Cross-Attention (Price x Events)")
    plt.legend(); plt.show()

plot_forecasts("GOLD", gold_df, gold_preds)
plot_forecasts("NIKKEI INVERSE", nikkei_df, nikkei_preds)

In [ ]:
# 9. News Database View (Last 10 Analyzed Headlines)
conn = sqlite3.connect(DB_NAME)
display(pd.read_sql_query("SELECT date, title, sentiment_score, topic FROM news_articles ORDER BY id DESC LIMIT 10", conn))
conn.close()
